# 2 Stages 4 layers 16 Batch Size

In [6]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from tqdm import tqdm
import ast
import joblib
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

class TestConfig:
    # Data configuration
    features_dim = 256
    num_classes = 2
    
    # Model architecture (must match training config)
    num_stages = 2
    num_layers = 4
    num_f_maps = 64
    kernel_size = 1
    dropout = 0.7
    
    # Paths (update these to match your setup)
    test_csv_path = '../../../ViT_augmented_labeled_test.csv'  # Path to your test data
    save_dir = 'saved_models_2S4L16BS'
    scaler_path = 'saved_models_2S4L16BS/scaler.save'
    model_path = 'saved_models_2S4L16BS/best_model.pth'
    results_dir = 'test_results_2S4L16BS'

    # Testing parameters
    batch_size = 16

def print_step(message, level=1):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    prefix = "  " * (level-1) + "» " if level > 1 else ""
    print(f"[{timestamp}] {prefix}{message}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print_step(f"Using device: {device}")

# Model Architecture (same as training)
class DilatedResidualLayer(nn.Module):
    def __init__(self, dilation, in_channels, out_channels, kernel_size, dropout):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        
        self.conv = nn.Sequential(
            nn.BatchNorm1d(in_channels),
            nn.GELU(),
            nn.Conv1d(in_channels, out_channels, kernel_size, 
                     padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 1),
            nn.Dropout(dropout)
        )
        self.skip = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        
    def forward(self, x):
        return self.conv(x) + self.skip(x)

class AV_MSTCN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Conv1d(config['features_dim'], config['num_f_maps'], 1),
            nn.BatchNorm1d(config['num_f_maps']),
            nn.GELU()
        )
        
        self.stages = nn.ModuleList([
            nn.Sequential(*[
                DilatedResidualLayer(2**i, config['num_f_maps'], config['num_f_maps'], 
                              config['kernel_size'], config['dropout'])
                for i in range(config['num_layers'])
            ]) for _ in range(config['num_stages'])
        ])
        
        self.attention = nn.Sequential(
            nn.Conv1d(config['num_f_maps'], config['num_f_maps']//4, 1),
            nn.GELU(),
            nn.Conv1d(config['num_f_maps']//4, 1, 1),
            nn.Softmax(dim=2)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(config['num_f_maps'], config['num_f_maps']//2),
            nn.LayerNorm(config['num_f_maps']//2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(config['num_f_maps']//2, config['num_classes'])
        )
        
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.input_proj(x)
        
        for stage in self.stages:
            x = stage(x)
        
        attn_weights = self.attention(x)
        x = torch.sum(x * attn_weights, dim=2)
        return self.classifier(x)

class AV_TestDataset(Dataset):
    def __init__(self, video_names, data_df, scaler):
        self.video_names = video_names
        self.data_df = data_df.groupby('Video File')
        self.label_map = {'fake': 0, 'real': 1}
        self.scaler = scaler
        
        print_step(f"Initializing test dataset with {len(video_names)} videos", level=2)
        
    def _get_features(self, video_name):
        video_data = self.data_df.get_group(video_name)
        features = np.stack([ast.literal_eval(x) if isinstance(x, str) else x 
                       for x in video_data['Features']])
        return features
    
    def __len__(self):
        return len(self.video_names)
    
    def __getitem__(self, idx):
        video_name = self.video_names[idx]
        features = self._get_features(video_name)
            
        features = self.scaler.transform(features)
        label = self.label_map[self.data_df.get_group(video_name)['label'].iloc[0].lower().strip()]
        
        return torch.FloatTensor(features), label, video_name

def safe_collate(batch):
    batch.sort(key=lambda x: x[0].shape[0], reverse=True)
    features, labels, video_names = zip(*batch)
    
    lengths = [f.shape[0] for f in features]
    max_len = max(lengths)
    padded_features = torch.zeros(len(batch), max_len, features[0].shape[1])
    for i, (f, l) in enumerate(zip(features, lengths)):
        padded_features[i, :l] = f
        
    return padded_features, torch.LongTensor(labels), video_names, torch.tensor(lengths)

def plot_confusion_matrix(cm, class_names, save_path):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.savefig(save_path)
    plt.close()

def save_metrics_report(metrics, save_path):
    with open(save_path, 'w') as f:
        json.dump(metrics, f, indent=4)

def load_model_and_scaler_forced():
    # Load scaler
    scaler = joblib.load(TestConfig.scaler_path)
    print_step(f"Loaded scaler from {TestConfig.scaler_path}", level=2)
    
    # Initialize model with the same architecture
    model = AV_MSTCN({
        'features_dim': TestConfig.features_dim,
        'num_classes': TestConfig.num_classes,
        'num_stages': TestConfig.num_stages,
        'num_layers': TestConfig.num_layers,
        'num_f_maps': TestConfig.num_f_maps,
        'kernel_size': TestConfig.kernel_size,
        'dropout': TestConfig.dropout
    }).to(device)
    
    # Load the checkpoint with forced weights
    checkpoint = torch.load(TestConfig.model_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Print model weights for verification
    print_step("Model weights summary:", level=2)
    for name, param in model.named_parameters():
        print_step(f"{name}: {param.shape}", level=3)
    
    return model, scaler

def test_model_forced():
    os.makedirs(TestConfig.results_dir, exist_ok=True)
    
    # Load data
    print_step("Loading test data...")
    df = pd.read_csv(TestConfig.test_csv_path)
    video_names = df['Video File'].unique()
    
    # Load model and scaler with forced weights
    model, scaler = load_model_and_scaler_forced()
    
    # Create test dataset and loader
    test_dataset = AV_TestDataset(video_names, df, scaler)
    test_loader = DataLoader(
        test_dataset,
        batch_size=TestConfig.batch_size,
        collate_fn=safe_collate,
        shuffle=False,
        num_workers=0
    )
    
    # Testing
    print_step("Starting testing...")
    all_labels = []
    all_probs = []
    all_preds = []
    video_results = []
    
    with torch.no_grad():
        for features, labels, names, _ in tqdm(test_loader, desc="Testing"):
            features = features.to(device)
            outputs = model(features)
            
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            
            # Store results per video
            for i, vid in enumerate(names):
                video_results.append({
                    'video_name': vid,
                    'true_label': 'fake' if labels[i].item() == 0 else 'real',
                    'predicted_label': 'fake' if preds[i] == 0 else 'real',
                    'fake_prob': probs[i][0].item(),
                    'real_prob': probs[i][1].item(),
                    'correct': int(preds[i] == labels[i].item())
                })
    
    # Calculate metrics
    print_step("Calculating metrics...")
    metrics = {
        'accuracy': 100 * accuracy_score(all_labels, all_preds),
        'balanced_accuracy': 100 * balanced_accuracy_score(all_labels, all_preds),
        'roc_auc': 100 * roc_auc_score(all_labels, np.array(all_probs)[:, 1]),
        'classification_report': classification_report(
            all_labels, 
            all_preds, 
            target_names=['fake', 'real'],
            digits=4,
            output_dict=True
        )
    }
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plot_confusion_matrix(cm, ['fake', 'real'], 
                         os.path.join(TestConfig.results_dir, 'confusion_matrix.png'))
    
    # Save results
    print_step("Saving results...")
    pd.DataFrame(video_results).to_csv(
        os.path.join(TestConfig.results_dir, 'video_predictions.csv'), 
        index=False
    )
    
    save_metrics_report(metrics, os.path.join(TestConfig.results_dir, 'test_metrics.json'))
    
    # Print summary
    print_step("\nTest Results Summary:", level=2)
    print_step(f"Accuracy: {metrics['accuracy']:.2f}%", level=3)
    print_step(f"Balanced Accuracy: {metrics['balanced_accuracy']:.2f}%", level=3)
    print_step(f"ROC AUC: {metrics['roc_auc']:.2f}%", level=3)
    
    print_step("\nClassification Report:", level=2)
    print(classification_report(
        all_labels, 
        all_preds, 
        target_names=['fake', 'real'],
        digits=4
    ))
    
    print_step(f"\nTest results saved to {TestConfig.results_dir}", level=2)

if __name__ == '__main__':
    test_model_forced()

[2025-07-04 10:30:47] Using device: cuda
[2025-07-04 10:30:47] Loading test data...
[2025-07-04 10:31:43]   » Loaded scaler from saved_models_2S4L16BS/scaler.save
[2025-07-04 10:31:43]   » Model weights summary:
[2025-07-04 10:31:43]     » input_proj.0.weight: torch.Size([64, 256, 1])
[2025-07-04 10:31:43]     » input_proj.0.bias: torch.Size([64])
[2025-07-04 10:31:43]     » input_proj.1.weight: torch.Size([64])
[2025-07-04 10:31:43]     » input_proj.1.bias: torch.Size([64])
[2025-07-04 10:31:43]     » stages.0.0.conv.0.weight: torch.Size([64])
[2025-07-04 10:31:43]     » stages.0.0.conv.0.bias: torch.Size([64])
[2025-07-04 10:31:43]     » stages.0.0.conv.2.weight: torch.Size([64, 64, 1])
[2025-07-04 10:31:43]     » stages.0.0.conv.2.bias: torch.Size([64])
[2025-07-04 10:31:43]     » stages.0.0.conv.3.weight: torch.Size([64])
[2025-07-04 10:31:43]     » stages.0.0.conv.3.bias: torch.Size([64])
[2025-07-04 10:31:43]     » stages.0.0.conv.6.weight: torch.Size([64, 64, 1])
[2025-07-04 10:

Testing: 100%|███████████████████████████████████████████████████████████████████████| 233/233 [21:47<00:00,  5.61s/it]


[2025-07-04 10:53:31] Calculating metrics...
[2025-07-04 10:53:31] Saving results...
[2025-07-04 10:53:31]   » 
Test Results Summary:
[2025-07-04 10:53:31]     » Accuracy: 99.49%
[2025-07-04 10:53:31]     » Balanced Accuracy: 99.13%
[2025-07-04 10:53:31]     » ROC AUC: 99.92%
[2025-07-04 10:53:31]   » 
Classification Report:
              precision    recall  f1-score   support

        fake     0.9946    0.9986    0.9966      2784
        real     0.9957    0.9841    0.9898       941

    accuracy                         0.9949      3725
   macro avg     0.9952    0.9913    0.9932      3725
weighted avg     0.9949    0.9949    0.9949      3725

[2025-07-04 10:53:31]   » 
Test results saved to test_results_2S4L16BS


# 2 Stages 3 layers 32 Batch Size

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from tqdm import tqdm
import ast
import joblib
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

class TestConfig:
    # Data configuration
    features_dim = 256
    num_classes = 2
    
    # Model architecture (must match training config)
    num_stages = 2
    num_layers = 3
    num_f_maps = 64
    kernel_size = 1
    dropout = 0.7
    
    # Paths (update these to match your setup)
    test_csv_path = '../../../ViT_augmented_labeled_test.csv'  # Path to your test data
    save_dir = 'saved_models_2S3L32BS'
    scaler_path = 'saved_models_2S3L32BS/scaler.save'
    model_path = 'saved_models_2S3L32BS/best_model.pth'
    results_dir = 'test_results_2S3L32BS'

    # Testing parameters
    batch_size = 16

def print_step(message, level=1):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    prefix = "  " * (level-1) + "» " if level > 1 else ""
    print(f"[{timestamp}] {prefix}{message}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print_step(f"Using device: {device}")

# Model Architecture (same as training)
class DilatedResidualLayer(nn.Module):
    def __init__(self, dilation, in_channels, out_channels, kernel_size, dropout):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        
        self.conv = nn.Sequential(
            nn.BatchNorm1d(in_channels),
            nn.GELU(),
            nn.Conv1d(in_channels, out_channels, kernel_size, 
                     padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 1),
            nn.Dropout(dropout)
        )
        self.skip = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        
    def forward(self, x):
        return self.conv(x) + self.skip(x)

class AV_MSTCN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Conv1d(config['features_dim'], config['num_f_maps'], 1),
            nn.BatchNorm1d(config['num_f_maps']),
            nn.GELU()
        )
        
        self.stages = nn.ModuleList([
            nn.Sequential(*[
                DilatedResidualLayer(2**i, config['num_f_maps'], config['num_f_maps'], 
                              config['kernel_size'], config['dropout'])
                for i in range(config['num_layers'])
            ]) for _ in range(config['num_stages'])
        ])
        
        self.attention = nn.Sequential(
            nn.Conv1d(config['num_f_maps'], config['num_f_maps']//4, 1),
            nn.GELU(),
            nn.Conv1d(config['num_f_maps']//4, 1, 1),
            nn.Softmax(dim=2)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(config['num_f_maps'], config['num_f_maps']//2),
            nn.LayerNorm(config['num_f_maps']//2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(config['num_f_maps']//2, config['num_classes'])
        )
        
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.input_proj(x)
        
        for stage in self.stages:
            x = stage(x)
        
        attn_weights = self.attention(x)
        x = torch.sum(x * attn_weights, dim=2)
        return self.classifier(x)

class AV_TestDataset(Dataset):
    def __init__(self, video_names, data_df, scaler):
        self.video_names = video_names
        self.data_df = data_df.groupby('Video File')
        self.label_map = {'fake': 0, 'real': 1}
        self.scaler = scaler
        
        print_step(f"Initializing test dataset with {len(video_names)} videos", level=2)
        
    def _get_features(self, video_name):
        video_data = self.data_df.get_group(video_name)
        features = np.stack([ast.literal_eval(x) if isinstance(x, str) else x 
                       for x in video_data['Features']])
        return features
    
    def __len__(self):
        return len(self.video_names)
    
    def __getitem__(self, idx):
        video_name = self.video_names[idx]
        features = self._get_features(video_name)
            
        features = self.scaler.transform(features)
        label = self.label_map[self.data_df.get_group(video_name)['label'].iloc[0].lower().strip()]
        
        return torch.FloatTensor(features), label, video_name

def safe_collate(batch):
    batch.sort(key=lambda x: x[0].shape[0], reverse=True)
    features, labels, video_names = zip(*batch)
    
    lengths = [f.shape[0] for f in features]
    max_len = max(lengths)
    padded_features = torch.zeros(len(batch), max_len, features[0].shape[1])
    for i, (f, l) in enumerate(zip(features, lengths)):
        padded_features[i, :l] = f
        
    return padded_features, torch.LongTensor(labels), video_names, torch.tensor(lengths)

def plot_confusion_matrix(cm, class_names, save_path):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.savefig(save_path)
    plt.close()

def save_metrics_report(metrics, save_path):
    with open(save_path, 'w') as f:
        json.dump(metrics, f, indent=4)

def load_model_and_scaler_forced():
    # Load scaler
    scaler = joblib.load(TestConfig.scaler_path)
    print_step(f"Loaded scaler from {TestConfig.scaler_path}", level=2)
    
    # Initialize model with the same architecture
    model = AV_MSTCN({
        'features_dim': TestConfig.features_dim,
        'num_classes': TestConfig.num_classes,
        'num_stages': TestConfig.num_stages,
        'num_layers': TestConfig.num_layers,
        'num_f_maps': TestConfig.num_f_maps,
        'kernel_size': TestConfig.kernel_size,
        'dropout': TestConfig.dropout
    }).to(device)
    
    # Load the checkpoint with forced weights
    checkpoint = torch.load(TestConfig.model_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Print model weights for verification
    print_step("Model weights summary:", level=2)
    for name, param in model.named_parameters():
        print_step(f"{name}: {param.shape}", level=3)
    
    return model, scaler

def test_model_forced():
    os.makedirs(TestConfig.results_dir, exist_ok=True)
    
    # Load data
    print_step("Loading test data...")
    df = pd.read_csv(TestConfig.test_csv_path)
    video_names = df['Video File'].unique()
    
    # Load model and scaler with forced weights
    model, scaler = load_model_and_scaler_forced()
    
    # Create test dataset and loader
    test_dataset = AV_TestDataset(video_names, df, scaler)
    test_loader = DataLoader(
        test_dataset,
        batch_size=TestConfig.batch_size,
        collate_fn=safe_collate,
        shuffle=False,
        num_workers=0
    )
    
    # Testing
    print_step("Starting testing...")
    all_labels = []
    all_probs = []
    all_preds = []
    video_results = []
    
    with torch.no_grad():
        for features, labels, names, _ in tqdm(test_loader, desc="Testing"):
            features = features.to(device)
            outputs = model(features)
            
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            
            # Store results per video
            for i, vid in enumerate(names):
                video_results.append({
                    'video_name': vid,
                    'true_label': 'fake' if labels[i].item() == 0 else 'real',
                    'predicted_label': 'fake' if preds[i] == 0 else 'real',
                    'fake_prob': probs[i][0].item(),
                    'real_prob': probs[i][1].item(),
                    'correct': int(preds[i] == labels[i].item())
                })
    
    # Calculate metrics
    print_step("Calculating metrics...")
    metrics = {
        'accuracy': 100 * accuracy_score(all_labels, all_preds),
        'balanced_accuracy': 100 * balanced_accuracy_score(all_labels, all_preds),
        'roc_auc': 100 * roc_auc_score(all_labels, np.array(all_probs)[:, 1]),
        'classification_report': classification_report(
            all_labels, 
            all_preds, 
            target_names=['fake', 'real'],
            digits=4,
            output_dict=True
        )
    }
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plot_confusion_matrix(cm, ['fake', 'real'], 
                         os.path.join(TestConfig.results_dir, 'confusion_matrix.png'))
    
    # Save results
    print_step("Saving results...")
    pd.DataFrame(video_results).to_csv(
        os.path.join(TestConfig.results_dir, 'video_predictions.csv'), 
        index=False
    )
    
    save_metrics_report(metrics, os.path.join(TestConfig.results_dir, 'test_metrics.json'))
    
    # Print summary
    print_step("\nTest Results Summary:", level=2)
    print_step(f"Accuracy: {metrics['accuracy']:.2f}%", level=3)
    print_step(f"Balanced Accuracy: {metrics['balanced_accuracy']:.2f}%", level=3)
    print_step(f"ROC AUC: {metrics['roc_auc']:.2f}%", level=3)
    
    print_step("\nClassification Report:", level=2)
    print(classification_report(
        all_labels, 
        all_preds, 
        target_names=['fake', 'real'],
        digits=4
    ))
    
    print_step(f"\nTest results saved to {TestConfig.results_dir}", level=2)

if __name__ == '__main__':
    test_model_forced()

[2025-07-15 18:27:28] Using device: cuda
[2025-07-15 18:27:28] Loading test data...
[2025-07-15 18:28:29]   » Loaded scaler from saved_models_2S3L32BS/scaler.save
[2025-07-15 18:28:29]   » Model weights summary:
[2025-07-15 18:28:29]     » input_proj.0.weight: torch.Size([64, 256, 1])
[2025-07-15 18:28:29]     » input_proj.0.bias: torch.Size([64])
[2025-07-15 18:28:29]     » input_proj.1.weight: torch.Size([64])
[2025-07-15 18:28:29]     » input_proj.1.bias: torch.Size([64])
[2025-07-15 18:28:29]     » stages.0.0.conv.0.weight: torch.Size([64])
[2025-07-15 18:28:29]     » stages.0.0.conv.0.bias: torch.Size([64])
[2025-07-15 18:28:29]     » stages.0.0.conv.2.weight: torch.Size([64, 64, 1])
[2025-07-15 18:28:29]     » stages.0.0.conv.2.bias: torch.Size([64])
[2025-07-15 18:28:29]     » stages.0.0.conv.3.weight: torch.Size([64])
[2025-07-15 18:28:29]     » stages.0.0.conv.3.bias: torch.Size([64])
[2025-07-15 18:28:29]     » stages.0.0.conv.6.weight: torch.Size([64, 64, 1])
[2025-07-15 18:

Testing: 100%|███████████████████████████████████████████████████████████████████████| 233/233 [21:28<00:00,  5.53s/it]


[2025-07-15 18:49:58] Calculating metrics...
[2025-07-15 18:49:58] Saving results...
[2025-07-15 18:49:58]   » 
Test Results Summary:
[2025-07-15 18:49:58]     » Accuracy: 95.79%
[2025-07-15 18:49:58]     » Balanced Accuracy: 91.66%
[2025-07-15 18:49:58]     » ROC AUC: 99.76%
[2025-07-15 18:49:58]   » 
Classification Report:
              precision    recall  f1-score   support

        fake     0.9466    1.0000    0.9726      2784
        real     1.0000    0.8332    0.9090       941

    accuracy                         0.9579      3725
   macro avg     0.9733    0.9166    0.9408      3725
weighted avg     0.9601    0.9579    0.9565      3725

[2025-07-15 18:49:58]   » 
Test results saved to test_results_2S3L32BS


# 2 stages 5 layers 64 batch size

In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from tqdm import tqdm
import ast
import joblib
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

class TestConfig:
    # Data configuration
    features_dim = 256
    num_classes = 2
    
    # Model architecture (must match training config)
    num_stages = 2
    num_layers = 5
    num_f_maps = 64
    kernel_size = 1
    dropout = 0.7
    
    # Paths (update these to match your setup)
    test_csv_path = '../../../ViT_augmented_labeled_test.csv'  # Path to your test data
    save_dir = 'saved_models_2S5L64BS'
    scaler_path = 'saved_models_2S5L64BS/scaler.save'
    model_path = 'saved_models_2S5L64BS/best_model.pth'
    results_dir = 'test_results_2S5L64BS'

    # Testing parameters
    batch_size = 16

def print_step(message, level=1):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    prefix = "  " * (level-1) + "» " if level > 1 else ""
    print(f"[{timestamp}] {prefix}{message}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print_step(f"Using device: {device}")

# Model Architecture (same as training)
class DilatedResidualLayer(nn.Module):
    def __init__(self, dilation, in_channels, out_channels, kernel_size, dropout):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        
        self.conv = nn.Sequential(
            nn.BatchNorm1d(in_channels),
            nn.GELU(),
            nn.Conv1d(in_channels, out_channels, kernel_size, 
                     padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 1),
            nn.Dropout(dropout)
        )
        self.skip = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        
    def forward(self, x):
        return self.conv(x) + self.skip(x)

class AV_MSTCN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Conv1d(config['features_dim'], config['num_f_maps'], 1),
            nn.BatchNorm1d(config['num_f_maps']),
            nn.GELU()
        )
        
        self.stages = nn.ModuleList([
            nn.Sequential(*[
                DilatedResidualLayer(2**i, config['num_f_maps'], config['num_f_maps'], 
                              config['kernel_size'], config['dropout'])
                for i in range(config['num_layers'])
            ]) for _ in range(config['num_stages'])
        ])
        
        self.attention = nn.Sequential(
            nn.Conv1d(config['num_f_maps'], config['num_f_maps']//4, 1),
            nn.GELU(),
            nn.Conv1d(config['num_f_maps']//4, 1, 1),
            nn.Softmax(dim=2)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(config['num_f_maps'], config['num_f_maps']//2),
            nn.LayerNorm(config['num_f_maps']//2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(config['num_f_maps']//2, config['num_classes'])
        )
        
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.input_proj(x)
        
        for stage in self.stages:
            x = stage(x)
        
        attn_weights = self.attention(x)
        x = torch.sum(x * attn_weights, dim=2)
        return self.classifier(x)

class AV_TestDataset(Dataset):
    def __init__(self, video_names, data_df, scaler):
        self.video_names = video_names
        self.data_df = data_df.groupby('Video File')
        self.label_map = {'fake': 0, 'real': 1}
        self.scaler = scaler
        
        print_step(f"Initializing test dataset with {len(video_names)} videos", level=2)
        
    def _get_features(self, video_name):
        video_data = self.data_df.get_group(video_name)
        features = np.stack([ast.literal_eval(x) if isinstance(x, str) else x 
                       for x in video_data['Features']])
        return features
    
    def __len__(self):
        return len(self.video_names)
    
    def __getitem__(self, idx):
        video_name = self.video_names[idx]
        features = self._get_features(video_name)
            
        features = self.scaler.transform(features)
        label = self.label_map[self.data_df.get_group(video_name)['label'].iloc[0].lower().strip()]
        
        return torch.FloatTensor(features), label, video_name

def safe_collate(batch):
    batch.sort(key=lambda x: x[0].shape[0], reverse=True)
    features, labels, video_names = zip(*batch)
    
    lengths = [f.shape[0] for f in features]
    max_len = max(lengths)
    padded_features = torch.zeros(len(batch), max_len, features[0].shape[1])
    for i, (f, l) in enumerate(zip(features, lengths)):
        padded_features[i, :l] = f
        
    return padded_features, torch.LongTensor(labels), video_names, torch.tensor(lengths)

def plot_confusion_matrix(cm, class_names, save_path):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.savefig(save_path)
    plt.close()

def save_metrics_report(metrics, save_path):
    with open(save_path, 'w') as f:
        json.dump(metrics, f, indent=4)

def load_model_and_scaler_forced():
    # Load scaler
    scaler = joblib.load(TestConfig.scaler_path)
    print_step(f"Loaded scaler from {TestConfig.scaler_path}", level=2)
    
    # Initialize model with the same architecture
    model = AV_MSTCN({
        'features_dim': TestConfig.features_dim,
        'num_classes': TestConfig.num_classes,
        'num_stages': TestConfig.num_stages,
        'num_layers': TestConfig.num_layers,
        'num_f_maps': TestConfig.num_f_maps,
        'kernel_size': TestConfig.kernel_size,
        'dropout': TestConfig.dropout
    }).to(device)
    
    # Load the checkpoint with forced weights
    checkpoint = torch.load(TestConfig.model_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Print model weights for verification
    print_step("Model weights summary:", level=2)
    for name, param in model.named_parameters():
        print_step(f"{name}: {param.shape}", level=3)
    
    return model, scaler

def test_model_forced():
    os.makedirs(TestConfig.results_dir, exist_ok=True)
    
    # Load data
    print_step("Loading test data...")
    df = pd.read_csv(TestConfig.test_csv_path)
    video_names = df['Video File'].unique()
    
    # Load model and scaler with forced weights
    model, scaler = load_model_and_scaler_forced()
    
    # Create test dataset and loader
    test_dataset = AV_TestDataset(video_names, df, scaler)
    test_loader = DataLoader(
        test_dataset,
        batch_size=TestConfig.batch_size,
        collate_fn=safe_collate,
        shuffle=False,
        num_workers=0
    )
    
    # Testing
    print_step("Starting testing...")
    all_labels = []
    all_probs = []
    all_preds = []
    video_results = []
    
    with torch.no_grad():
        for features, labels, names, _ in tqdm(test_loader, desc="Testing"):
            features = features.to(device)
            outputs = model(features)
            
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            
            # Store results per video
            for i, vid in enumerate(names):
                video_results.append({
                    'video_name': vid,
                    'true_label': 'fake' if labels[i].item() == 0 else 'real',
                    'predicted_label': 'fake' if preds[i] == 0 else 'real',
                    'fake_prob': probs[i][0].item(),
                    'real_prob': probs[i][1].item(),
                    'correct': int(preds[i] == labels[i].item())
                })
    
    # Calculate metrics
    print_step("Calculating metrics...")
    metrics = {
        'accuracy': 100 * accuracy_score(all_labels, all_preds),
        'balanced_accuracy': 100 * balanced_accuracy_score(all_labels, all_preds),
        'roc_auc': 100 * roc_auc_score(all_labels, np.array(all_probs)[:, 1]),
        'classification_report': classification_report(
            all_labels, 
            all_preds, 
            target_names=['fake', 'real'],
            digits=4,
            output_dict=True
        )
    }
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plot_confusion_matrix(cm, ['fake', 'real'], 
                         os.path.join(TestConfig.results_dir, 'confusion_matrix.png'))
    
    # Save results
    print_step("Saving results...")
    pd.DataFrame(video_results).to_csv(
        os.path.join(TestConfig.results_dir, 'video_predictions.csv'), 
        index=False
    )
    
    save_metrics_report(metrics, os.path.join(TestConfig.results_dir, 'test_metrics.json'))
    
    # Print summary
    print_step("\nTest Results Summary:", level=2)
    print_step(f"Accuracy: {metrics['accuracy']:.2f}%", level=3)
    print_step(f"Balanced Accuracy: {metrics['balanced_accuracy']:.2f}%", level=3)
    print_step(f"ROC AUC: {metrics['roc_auc']:.2f}%", level=3)
    
    print_step("\nClassification Report:", level=2)
    print(classification_report(
        all_labels, 
        all_preds, 
        target_names=['fake', 'real'],
        digits=4
    ))
    
    print_step(f"\nTest results saved to {TestConfig.results_dir}", level=2)

if __name__ == '__main__':
    test_model_forced()

[2025-07-16 11:13:59] Using device: cuda
[2025-07-16 11:13:59] Loading test data...
[2025-07-16 11:14:37]   » Loaded scaler from saved_models_2S5L64BS/scaler.save
[2025-07-16 11:14:37]   » Model weights summary:
[2025-07-16 11:14:37]     » input_proj.0.weight: torch.Size([64, 256, 1])
[2025-07-16 11:14:37]     » input_proj.0.bias: torch.Size([64])
[2025-07-16 11:14:37]     » input_proj.1.weight: torch.Size([64])
[2025-07-16 11:14:37]     » input_proj.1.bias: torch.Size([64])
[2025-07-16 11:14:37]     » stages.0.0.conv.0.weight: torch.Size([64])
[2025-07-16 11:14:37]     » stages.0.0.conv.0.bias: torch.Size([64])
[2025-07-16 11:14:37]     » stages.0.0.conv.2.weight: torch.Size([64, 64, 1])
[2025-07-16 11:14:37]     » stages.0.0.conv.2.bias: torch.Size([64])
[2025-07-16 11:14:37]     » stages.0.0.conv.3.weight: torch.Size([64])
[2025-07-16 11:14:37]     » stages.0.0.conv.3.bias: torch.Size([64])
[2025-07-16 11:14:37]     » stages.0.0.conv.6.weight: torch.Size([64, 64, 1])
[2025-07-16 11:

Testing: 100%|███████████████████████████████████████████████████████████████████████| 233/233 [15:44<00:00,  4.05s/it]


[2025-07-16 11:30:22] Calculating metrics...
[2025-07-16 11:30:22] Saving results...
[2025-07-16 11:30:22]   » 
Test Results Summary:
[2025-07-16 11:30:22]     » Accuracy: 87.25%
[2025-07-16 11:30:22]     » Balanced Accuracy: 74.83%
[2025-07-16 11:30:22]     » ROC AUC: 98.36%
[2025-07-16 11:30:22]   » 
Classification Report:
              precision    recall  f1-score   support

        fake     0.8547    0.9993    0.9213      2784
        real     0.9957    0.4973    0.6634       941

    accuracy                         0.8725      3725
   macro avg     0.9252    0.7483    0.7924      3725
weighted avg     0.8903    0.8725    0.8562      3725

[2025-07-16 11:30:22]   » 
Test results saved to test_results_2S5L64BS
